# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

## 1. Config

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ── a. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.7 MB/s eta 0:00:00


In [3]:
# ── b. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust these paths to match your local environment
DATA_DIR   = Path("/content/drive/MyDrive/dl_final/")

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
IMG_SIZE        = 336

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA A100-SXM4-80GB


In [4]:
print("Global IMG_SIZE =", IMG_SIZE)

if "train_ds" in globals():
    print("train_ds.img_size =", train_ds.img_size)
    print("train_ds[0] image size =", train_ds[0]["image"].size)

if "train_lora_ds" in globals():
    print("train_lora_ds.img_size =", train_lora_ds.img_size)
    print("train_lora_ds[0] image size =", train_lora_ds[0]["image"].size)

if "val_lora_ds" in globals():
    print("val_lora_ds.img_size =", val_lora_ds.img_size)
    print("val_lora_ds[0] image size =", val_lora_ds[0]["image"].size)

Global IMG_SIZE = 336


## 2. Load and Preprocess Data

In [5]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

In [6]:
splits = {
    "train": train_df,
    "val": val_df,
    "test": test_df,
}

overview = []

for name, df in splits.items():
    overview.append({
        "split": name,
        "num_samples": len(df),
        "num_columns": df.shape[1],
        "has_answer": "answer" in df.columns,
        "has_solution": "solution" in df.columns,
    })

overview_df = pd.DataFrame(overview)
display(overview_df)

print("Train columns:")
print(train_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

,split,num_samples,num_columns,has_answer,has_solution
0,train,3109,15,True,True
1,val,1048,15,True,True
2,test,1008,13,False,False


Train columns:
['id', 'image_path', 'question', 'choices', 'num_choices', 'answer', 'hint', 'lecture', 'solution', 'task', 'grade', 'subject', 'topic', 'category', 'skill']

Test columns:
['id', 'image_path', 'question', 'choices', 'num_choices', 'hint', 'lecture', 'task', 'grade', 'subject', 'topic', 'category', 'skill']


In [7]:
important_cols = ["image_path", "question", "choices", "num_choices", "answer",
                  "hint", "lecture", "solution", "task", "grade", "subject",
                  "topic", "category", "skill"]

missing_rows = []

for name, df in splits.items():
    for col in important_cols:
        if col in df.columns:
            missing_rows.append({
                "split": name,
                "column": col,
                "missing_count": df[col].isna().sum(),
                "missing_ratio": df[col].isna().mean(),
            })

missing_df = pd.DataFrame(missing_rows)
missing_df = missing_df[missing_df["missing_count"] > 0].sort_values(
    ["split", "missing_ratio"], ascending=[True, False]
)

display(missing_df)

,split,column,missing_count,missing_ratio
32,test,hint,214,0.212302
33,test,lecture,183,0.181548
5,train,hint,724,0.232872
7,train,solution,529,0.170151
6,train,lecture,440,0.141525
19,val,hint,232,0.221374
21,val,solution,172,0.164122
20,val,lecture,133,0.126908


In [8]:
def normalized_counts(df, col, topn=None):
    vc = df[col].value_counts(normalize=True)
    if topn is not None:
        vc = vc.head(topn)
    return vc

# num_choices distribution
num_choice_summary = pd.DataFrame({
    name: df["num_choices"].value_counts().sort_index()
    for name, df in splits.items()
}).fillna(0).astype(int)

display(num_choice_summary)

# answer distribution for labeled splits
answer_summary = pd.DataFrame({
    name: df["answer"].value_counts().sort_index()
    for name, df in {"train": train_df, "val": val_df}.items()
}).fillna(0).astype(int)

display(answer_summary)

,train,val,test
num_choices,,,
2,664,244,272
3,1552,508,438
4,783,252,260
5,110,44,38


,train,val
answer,,
0,1124,347
1,1028,372
2,737,247
3,204,75
4,16,7


In [9]:
train_small = train_df.sample(n=1000, random_state=SEED)
val_small = val_df.sample(n=500, random_state=SEED)

print(f"Saved train_small: {len(train_small)}")
print(f"Saved val_small: {len(val_small)}")

Saved train_small: 1000
Saved val_small: 500


## 3. Prompt Engineering

In [ ]:
# -- 3a. prompt config
BEST_USE_HINT = True
BEST_USE_LECTURE = False
BEST_LECTURE_TRUNC_CHARS = 0
BEST_ANSWER_PHRASE = "Answer (single letter):"
BEST_CHOICE_STYLE = "dot"
BEST_USE_METADATA = False
BEST_METADATA_COLS = ["subject", "topic"]
BEST_USE_SOLUTION_TRAIN_ONLY = False

In [ ]:
# ── 3b. Choice Shuffling Augmentation
import json
import ast

CHOICE_LETTERS = "ABCDEFGHIJ"

def ensure_choices(x):
    if isinstance(x, list):
        return x

    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            try:
                return ast.literal_eval(x)
            except Exception as e:
                raise ValueError(f"Cannot parse choices: {x}") from e

    return list(x)

def _get_text_field(row, field):
    val = row.get(field, "")
    if pd.notna(val) and str(val).strip():
        return str(val).strip()
    return ""

def build_metadata(row, cols=BEST_METADATA_COLS):
    parts = []

    for col in cols:
        if col in row:
            val = row.get(col, "")
            if pd.notna(val) and str(val).strip():
                pretty_col = col.replace("_", " ").title()
                parts.append(f"{pretty_col}: {str(val).strip()}")

    return "\n".join(parts)

def shuffle_choices_and_answer(choices, answer, rng=None):
    # Randomly permute choices and update the gold answer index.
    if rng is None:
        rng = random

    choices = list(choices)
    answer = int(answer)
    n = len(choices)

    if n <= 1:
        return choices, answer, list(range(n))

    perm = list(range(n))
    rng.shuffle(perm)

    shuffled_choices = [choices[i] for i in perm]
    shuffled_answer = perm.index(answer)

    return shuffled_choices, shuffled_answer, perm

def build_prompt_from_parts(
    question,
    choices,
    answer=None,
    hint="",
    lecture="",
    metadata="",
    solution="",
    include_answer=False,
    use_hint=BEST_USE_HINT,
    use_lecture=BEST_USE_LECTURE,
    lecture_trunc_chars=BEST_LECTURE_TRUNC_CHARS,
    use_metadata=BEST_USE_METADATA,
    use_solution_train_only=BEST_USE_SOLUTION_TRAIN_ONLY,
    answer_phrase=BEST_ANSWER_PHRASE,
    choice_style=BEST_CHOICE_STYLE,
):

    # ----- format choices -----
    if choice_style == "dot":
        choices_str = "\n".join(
            f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
        )
    elif choice_style == "paren":
        choices_str = "\n".join(
            f"  ({CHOICE_LETTERS[i]}) {c}" for i, c in enumerate(choices)
        )
    else:
        raise ValueError(f"Unknown choice_style: {choice_style}")

    prompt = "<image>\n"

    # ----- metadata -----
    if use_metadata and metadata:
        prompt += f"Metadata:\n{metadata}\n\n"

    # ----- context: lecture + hint -----
    context_parts = []

    if use_lecture and lecture:
        lec = str(lecture).strip()
        if lecture_trunc_chars > 0 and len(lec) > lecture_trunc_chars:
            lec = lec[:lecture_trunc_chars].rsplit(" ", 1)[0] + "..."
        context_parts.append(lec)

    if use_hint and hint:
        context_parts.append(str(hint).strip())

    if context_parts:
        prompt += "Context:\n" + "\n\n".join(context_parts) + "\n\n"

    # ----- solution: train-only -----
    if include_answer and use_solution_train_only and solution:
        prompt += "Training solution:\n"
        prompt += str(solution).strip() + "\n\n"

    # ----- question + choices + answer -----
    prompt += f"Question: {question}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += answer_phrase

    if include_answer:
        if answer is None:
            raise ValueError("include_answer=True requires answer.")
        prompt += f" {CHOICE_LETTERS[int(answer)]}"

    return prompt

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    """
    Default unshuffled prompt for validation/test/inference.
    This does NOT shuffle choices.
    """

    metadata = build_metadata(row)
    solution = _get_text_field(row, "solution")

    return build_prompt_from_parts(
        question=row["question"],
        choices=ensure_choices(row["choices"]),
        answer=int(row["answer"]) if include_answer and "answer" in row else None,
        hint=_get_text_field(row, "hint"),
        lecture=_get_text_field(row, "lecture"),
        metadata=metadata,
        solution=solution,
        include_answer=include_answer,
    )

In [ ]:
# ── 3c. PyTorch Dataset ───────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        data_dir: Path,
        img_size: int = 224,
        is_train: bool = True,
        shuffle_choices: bool = False,
    ):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train
        self.shuffle_choices = shuffle_choices

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        p = self.data_dir / rel_path

        if not p.exists():
            p = self.data_dir / "images" / rel_path

        if not p.exists():
            raise FileNotFoundError(f"Image not found: {rel_path}, tried {p}")

        img = Image.open(p).convert("RGB")

        # padding resize
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        # img = resize_with_padding(img, self.img_size)

        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        choices = ensure_choices(row["choices"])
        answer = int(row["answer"]) if "answer" in row and pd.notna(row["answer"]) else -1
        perm = list(range(len(choices)))

        # Only training data should be shuffled.
        if self.is_train and self.shuffle_choices:
            choices, answer, perm = shuffle_choices_and_answer(choices, answer)

        metadata = build_metadata(row)

        text = build_prompt_from_parts(
            question=row["question"],
            choices=choices,
            answer=answer if self.is_train else None,
            hint=_get_text_field(row, "hint"),
            lecture=_get_text_field(row, "lecture"),
            metadata=metadata,
            solution=_get_text_field(row, "solution"),
            include_answer=self.is_train,
        )

        item = {
            "image": img,
            "text": text,
            "answer": answer,
            "choices": choices,
            "perm": perm,
            "id": row["id"] if "id" in row else idx,
        }
        return item


# Create datasets.
# IMPORTANT: shuffle_choices=True only for train.
train_ds = ScienceQADataset(
    train_df, DATA_DIR, img_size=IMG_SIZE, is_train=True, shuffle_choices=True
)
val_ds = ScienceQADataset(
    val_df, DATA_DIR, img_size=IMG_SIZE, is_train=False, shuffle_choices=False
)
test_ds = ScienceQADataset(
    test_df, DATA_DIR, img_size=IMG_SIZE, is_train=False, shuffle_choices=False
)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")
print("✓ Choice shuffling is ON for training and OFF for validation/test.")


Datasets created: train=3109, val=1048, test=1008
✓ Choice shuffling is ON for training and OFF for validation/test.


In [ ]:
print("Current Prompt Config:")
print("BEST_USE_HINT =", BEST_USE_HINT)
print("BEST_USE_LECTURE =", BEST_USE_LECTURE)
print("BEST_LECTURE_TRUNC_CHARS =", BEST_LECTURE_TRUNC_CHARS)
print("BEST_USE_METADATA =", BEST_USE_METADATA)
print("BEST_METADATA_COLS =", BEST_METADATA_COLS)
print("BEST_USE_SOLUTION_TRAIN_ONLY =", BEST_USE_SOLUTION_TRAIN_ONLY)
print("BEST_ANSWER_PHRASE =", BEST_ANSWER_PHRASE)
print("BEST_CHOICE_STYLE =", BEST_CHOICE_STYLE)

row = train_df.iloc[0]

print("\n" + "=" * 80)
print("TRAINING PROMPT EXAMPLE")
print("=" * 80)
print(build_prompt(row, include_answer=True))

print("\n" + "=" * 80)
print("VALIDATION / TEST PROMPT EXAMPLE")
print("=" * 80)
print(build_prompt(row, include_answer=False))

Current Prompt Config:
BEST_USE_HINT = True
BEST_USE_LECTURE = False
BEST_LECTURE_TRUNC_CHARS = 0
BEST_USE_METADATA = False
BEST_METADATA_COLS = ['subject', 'topic']
BEST_USE_SOLUTION_TRAIN_ONLY = False
BEST_ANSWER_PHRASE = Answer (single letter):
BEST_CHOICE_STYLE = dot

TRAINING PROMPT EXAMPLE
<image>
Context:
Animals often behave in certain ways that can increase their reproductive success. Read the passage about a specific animal behavior. Then, follow the instructions below.

Amazonian poison frogs live in tropical forests in northern South America. After a male and female frog mate, the female frog lays eggs on a plant. When tadpoles hatch from the eggs, the male frog lets the tadpoles climb onto his back. The male then searches for water trapped in the spaces where plants' leaves meet their stems. He puts his tadpoles in these small pools of water.
If the male frog puts a tadpole into a pool with a larger tadpole, the smaller tadpole is often eaten. So, the male frog usually put

## 4. Model Loading

This section loads `HuggingFaceTB/SmolVLM-500M-Instruct` and runs a quick inference example on one validation sample.

In [ ]:
# -- 4a. Reset model without restarting runtime ─────────────────────
import gc
import torch

for name in [
    "trainer",
    "model",
    "optimizer",
    "lr_scheduler",
    "train_result",
    "inputs",
    "generated_ids",
]:
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Old model/trainer cleared. Now rerun 3a to load a fresh model.")

Old model/trainer cleared. Now rerun 3a to load a fresh model.


In [ ]:
# ── 4b. Load SmolVLM model + run one inference example ───────────────────────
from transformers import AutoProcessor, AutoModelForVision2Seq

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
 )
if not torch.cuda.is_available():
    model.to(device)
model.eval()

# Pick a sample from validation set
sample = val_df.iloc[0]
sample_image = Image.open(DATA_DIR / sample["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample_prompt = build_prompt(sample, include_answer=False)

inputs = processor(
    text=[sample_prompt],
    images=[sample_image],
    return_tensors="pt",
)
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
    )

decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Prompt:")
print(sample_prompt)
print("\nModel output:")
print(decoded)
print(f"\nGround-truth answer index: {sample['answer']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Prompt:
<image>
Context:
Animals often behave in certain ways that can increase their reproductive success. Read the passage about a specific animal behavior. Then, follow the instructions below.

The snail leech is a type of worm that often lives in freshwater streams. After  reproduces, it attaches its eggs to a rock at the bottom of the stream. The leech then flattens its body over its eggs to protect them. The leech protects its eggs until they hatch, which takes four to seven days. During this time, the leech does not leave the eggs or eat.
Water snails are predators that eat leech eggs. The snails easily eat eggs that are not covered by an adult leech. But snails cannot easily get to eggs that are covered by a leech.
Figure: a snail leech.

Question: Why might covering its eggs with its body increase the reproductive success of a snail leech? Complete the claim below that answers this question and is best supported by the passage.
Covering its eggs with its body increases the cha

## 5. Log Likelihood Scoring (Single-Forward Optimized)

We use an optimized **single-forward** approach:
- Forward only up to "Answer:"
- Compare next-token probabilities for A/B/C/D  
- Select the choice with highest probability
- **~7x faster** than standard multi-forward approach

This method has been validated to give **identical accuracy** to the standard approach.

---

In [ ]:
# ── 5a. Verify Answer Letter Tokenization ────────────────────────────────────

def get_choice_token_ids(processor, choice_letters=CHOICE_LETTERS):
    """
    Get token IDs for choice letters (A, B, C, D, etc.).
    Returns dict mapping letter -> token_id.
    """
    tokenizer = processor.tokenizer
    choice_token_ids = {}

    for letter in choice_letters:
        # Try space-prefixed version first (usually single token)
        tokens_space = tokenizer.encode(" " + letter, add_special_tokens=False)
        tokens_plain = tokenizer.encode(letter, add_special_tokens=False)

        print(f"  {letter}: plain={tokens_plain}, space-prefix={tokens_space}")

        if len(tokens_space) == 1:
            choice_token_ids[letter] = tokens_space[0]
        elif len(tokens_plain) == 1:
            choice_token_ids[letter] = tokens_plain[0]
        else:
            raise ValueError(
                f"Letter '{letter}' cannot be represented as single token.\n"
                f"Cannot use single-forward optimization."
            )

    print("✓ All letters are single tokens")
    return choice_token_ids

# Get token IDs
CHOICE_TOKEN_IDS = get_choice_token_ids(processor)
print("\n✓ Ready for single-forward inference")


  A: plain=[49], space-prefix=[330]
  B: plain=[50], space-prefix=[389]
  C: plain=[51], space-prefix=[340]
  D: plain=[52], space-prefix=[422]
  E: plain=[53], space-prefix=[414]
  F: plain=[54], space-prefix=[426]
  G: plain=[55], space-prefix=[452]
  H: plain=[56], space-prefix=[407]
  I: plain=[57], space-prefix=[339]
  J: plain=[58], space-prefix=[530]
✓ All letters are single tokens

✓ Ready for single-forward inference


In [ ]:
# ── 5b. Single-Forward Batched Scoring ───────────────────────────────────────

import torch.nn.functional as F
from typing import List
from tqdm import tqdm  # Import here to avoid NameError

def score_samples_batched(
    model,
    processor,
    samples: List[pd.Series],
    data_dir: Path,
    choice_token_ids: dict,
    img_size: int = 224,
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
) -> List[int]:
    """
    Score samples using single-forward approach.
    Forward only up to "Answer:" and compare next-token probabilities.
    """
    images = []
    prompts = []
    num_choices_list = []

    for sample in samples:
        # Load image
        image = Image.open(data_dir / sample["image_path"]).convert("RGB")
        image = image.resize((img_size, img_size), Image.BICUBIC)
        images.append(image)

        # Build prompt (ends with "Answer:")
        prompt = build_prompt(sample, include_answer=False)
        prompts.append(prompt)

        num_choices_list.append(len(sample["choices"]))

    # Batch tokenize
    inputs = processor(
        text=prompts,
        images=images,
        return_tensors="pt",
        padding=True,
    )
    inputs = {k: v.to(device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    # Single forward pass
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    log_probs = F.log_softmax(logits, dim=-1)

    # Extract predictions
    predictions = []

    for i, num_choices in enumerate(num_choices_list):
        # Find last non-padding position
        input_ids = inputs["input_ids"][i]
        last_pos = (input_ids != processor.tokenizer.pad_token_id).sum().item() - 1

        # Get log probs at last position
        last_log_probs = log_probs[i, last_pos, :]

        # Get log probs for each choice
        choice_letters = [CHOICE_LETTERS[j] for j in range(num_choices)]
        choice_log_probs = [
            last_log_probs[choice_token_ids[letter]].item()
            for letter in choice_letters
        ]

        predicted_idx = int(np.argmax(choice_log_probs))
        predictions.append(predicted_idx)

    return predictions


def evaluate_batched(
    model,
    processor,
    eval_df: pd.DataFrame,
    data_dir: Path,
    choice_token_ids: dict,
    img_size: int = 224,
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    batch_size: int = 16,
    max_samples: int = None,
) -> dict:
    """Evaluate using single-forward batched approach."""
    model.eval()

    if max_samples is not None:
        eval_df = eval_df.head(max_samples)

    predictions = []
    ground_truth = []

    num_batches = (len(eval_df) + batch_size - 1) // batch_size

    for batch_idx in tqdm(range(num_batches), desc="Evaluating"):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, len(eval_df))

        batch_samples = [eval_df.iloc[i] for i in range(start_idx, end_idx)]

        batch_preds = score_samples_batched(
            model=model,
            processor=processor,
            samples=batch_samples,
            data_dir=data_dir,
            choice_token_ids=choice_token_ids,
            img_size=img_size,
            device=device,
        )

        predictions.extend(batch_preds)
        ground_truth.extend([int(s["answer"]) for s in batch_samples])

    correct = sum([p == g for p, g in zip(predictions, ground_truth)])
    total = len(predictions)
    accuracy = correct / total if total > 0 else 0.0

    return {
        "predictions": predictions,
        "ground_truth": ground_truth,
        "accuracy": accuracy,
        "correct": correct,
        "total": total,
    }

print("✓ Single-forward functions defined")


✓ Single-forward functions defined


### -- Cleaned: Prompt tuning tests and error analysis

## 6. Training with LORA

### 路线 A（重新训练）：按顺序跑 6a → 6b → 6c → 6d → 6e
### 路线 B（加载已有模型）：跑6f




In [ ]:
# ── 6a. Apply LoRA / DoRA adapters to the loaded SmolVLM ───────────
import inspect
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
MAX_TRAINABLE_PARAMS = 5_000_000
USE_DORA = False

LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]


def count_trainable_params(model):
    trainable = 0
    total = 0
    for _, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    print(f"Trainable params: {trainable:,}")
    print(f"Total params:     {total:,}")
    print(f"Trainable ratio:  {100 * trainable / total:.4f}%")
    return trainable, total


def show_matching_lora_modules(model, targets):
    matches = []
    for name, _ in model.named_modules():
        if any(name.endswith(t) for t in targets):
            matches.append(name)
    print(f"Found {len(matches)} modules matching LoRA targets.")
    return matches


# Prevent duplicate LoRA wrapping if you accidentally rerun this cell.
if hasattr(model, "peft_config") and len(getattr(model, "peft_config", {})) > 0:
    print("LoRA/PEFT adapter is already attached to this model. Skipping get_peft_model(...).")
    count_trainable_params(model)
else:
    model.config.use_cache = False

    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    # Required/recommended for k-bit training. It also freezes base parameters.
    # If the model is not quantized, this is still usually safe for LoRA training.
    model = prepare_model_for_kbit_training(model)

    print("Checking target modules before attaching LoRA...")
    _ = show_matching_lora_modules(model, LORA_TARGET_MODULES)

    lora_kwargs = dict(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=LORA_TARGET_MODULES,
    )

    # DoRA is available only in newer PEFT versions. Enable it when supported.
    if "use_dora" in inspect.signature(LoraConfig.__init__).parameters:
        lora_kwargs["use_dora"] = USE_DORA
        print(f"DoRA enabled: use_dora={USE_DORA}")
    else:
        print("This PEFT version does not support use_dora. Using standard LoRA.")

    lora_config = LoraConfig(**lora_kwargs)
    model = get_peft_model(model, lora_config)

    print("\nLoRA attached. Parameter check:")
    trainable, total = count_trainable_params(model)

    if trainable > MAX_TRAINABLE_PARAMS:
        print("\nWARNING: trainable parameters exceed 5M.")
        print("Reduce LORA_R from 8 to 4 and rerun from a fresh model load.")
    else:
        print("\nOK: trainable parameters are within the 5M budget.")


Checking target modules before attaching LoRA...
Found 260 modules matching LoRA targets.
DoRA enabled: use_dora=False

LoRA attached. Parameter check:
Trainable params: 4,784,128
Total params:     512,266,432
Trainable ratio:  0.9339%

OK: trainable parameters are within the 5M budget.


In [ ]:
# ── 6b. Data collator for SmolVLM LoRA training
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class PixPredDataCollatorForVLM:
    processor: Any
    max_length: int = 2048

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        images = [f["image"] for f in features]
        full_texts = [f["text"] for f in features]

        # Remove the final answer letter to get the prompt-only prefix.
        prefix_texts = []
        for f in features:
            answer_letter = CHOICE_LETTERS[int(f["answer"])]
            suffix = " " + answer_letter
            if f["text"].endswith(suffix):
                prefix_texts.append(f["text"][:-len(suffix)])
            else:
                prefix_texts.append(f["text"].rsplit(" ", 1)[0])

        full_inputs = self.processor(
            text=full_texts,
            images=images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length,
        )

        prefix_inputs = self.processor(
            text=prefix_texts,
            images=images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length,
        )

        labels = full_inputs["input_ids"].clone()
        pad_id = self.processor.tokenizer.pad_token_id

        # Always ignore padding tokens.
        labels[labels == pad_id] = -100

        # Ignore the full prompt; keep only answer-token labels.
        for i in range(labels.shape[0]):
            prefix_len = int((prefix_inputs["input_ids"][i] != pad_id).sum().item())
            full_len = int((full_inputs["input_ids"][i] != pad_id).sum().item())

            # Normal case: answer begins at prefix_len.
            if prefix_len < full_len:
                labels[i, :prefix_len] = -100
            else:
                # If truncation removed the answer, train only the last non-pad token as fallback.
                labels[i, :] = -100
                if full_len > 0:
                    labels[i, full_len - 1] = full_inputs["input_ids"][i, full_len - 1]

        full_inputs["labels"] = labels
        return full_inputs


collator = PixPredDataCollatorForVLM(processor=processor, max_length=2048)
print("✓ LoRA data collator ready")


✓ LoRA data collator ready


In [ ]:
# ── 6c. Build LoRA training datasets ────────────────────────────────────────
USE_SMALL_DEBUG_SET = False

SHUFFLE_AUG_REPEAT = 2

if USE_SMALL_DEBUG_SET:
    train_lora_df = train_small
    val_lora_df = val_small
else:
    train_lora_df = pd.concat(
        [train_df.copy() for _ in range(SHUFFLE_AUG_REPEAT)],
        ignore_index=True
    ).reset_index(drop=True)

    val_lora_df = val_df

train_lora_ds = ScienceQADataset(
    train_lora_df,
    DATA_DIR,
    img_size=IMG_SIZE,
    is_train=True,
    shuffle_choices=True,
)

val_lora_ds = ScienceQADataset(
    val_lora_df,
    DATA_DIR,
    img_size=IMG_SIZE,
    is_train=False,
    shuffle_choices=False,
)

print(f"LoRA train samples: {len(train_lora_ds)}")
print(f"LoRA val samples:   {len(val_lora_ds)}")
print("Example training text:")
print(train_lora_ds[0]["text"][:900])


LoRA train samples: 6218
LoRA val samples:   1048
Example training text:
<image>
Context:
Animals often behave in certain ways that can increase their reproductive success. Read the passage about a specific animal behavior. Then, follow the instructions below.

Amazonian poison frogs live in tropical forests in northern South America. After a male and female frog mate, the female frog lays eggs on a plant. When tadpoles hatch from the eggs, the male frog lets the tadpoles climb onto his back. The male then searches for water trapped in the spaces where plants' leaves meet their stems. He puts his tadpoles in these small pools of water.
If the male frog puts a tadpole into a pool with a larger tadpole, the smaller tadpole is often eaten. So, the male frog usually puts each tadpole into a pool of water that does not have other tadpoles in it. Each tadpole lives in its own pool until it undergoes metamorphosis to develop into a frog.
Figure: an Amazonian poison 


In [ ]:
# ── 6d. TrainingArguments + Trainer ─────────────────────────────────────────
from transformers import TrainingArguments, Trainer

LORA_OUTPUT_DIR = str(DATA_DIR / "hintonly4")

training_args = TrainingArguments(
    output_dir=LORA_OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="no",
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_lora_ds,
    data_collator=collator,
)

print("✓ Trainer ready")
print("Next cell will start LoRA training.")
print(f"Output directory: {LORA_OUTPUT_DIR}")


The model is already on multiple devices. Skipping the move to device specified in `args`.


✓ Trainer ready
Next cell will start LoRA training.
Output directory: /content/drive/MyDrive/dl_final/hintonly


In [ ]:
# ── 6e. Train LoRA adapter ──────────────────────────────────────────────────

model.train()
train_result = trainer.train()
print(train_result)

# Save only the LoRA adapter + processor/tokenizer.
trainer.model.save_pretrained(LORA_OUTPUT_DIR)
processor.save_pretrained(LORA_OUTPUT_DIR)

trainer.save_state()

print(f"✓ Saved LoRA adapter to: {LORA_OUTPUT_DIR}")


Step,Training Loss
20,2.090500
40,1.037300
60,0.657100
80,0.704500
100,0.607800
120,0.657600
140,0.493600
160,0.587800
180,0.542000
200,0.536000


Step,Training Loss
20,2.090500
40,1.037300
60,0.657100
80,0.704500
100,0.607800
120,0.657600
140,0.493600
160,0.587800
180,0.542000
200,0.536000


TrainOutput(global_step=778, training_loss=0.4803901307074147, metrics={'train_runtime': 17069.1501, 'train_samples_per_second': 0.729, 'train_steps_per_second': 0.046, 'total_flos': 4.77943586541504e+16, 'train_loss': 0.4803901307074147, 'epoch': 2.0})
✓ Saved LoRA adapter to: /content/drive/MyDrive/dl_final/hintonly


## 7. Validation

In [ ]:
# ── 7a. Full validation
from tqdm import tqdm
import hashlib

model.eval()

K_SHUFFLES = 3
ORIGINAL_BATCH_SIZE = 4

def stable_seed_from_id(x, base_seed=42):
    s = str(x).encode("utf-8")
    h = hashlib.md5(s).hexdigest()
    return base_seed + int(h[:8], 16)

def load_image_safe(rel_path):
    p = DATA_DIR / rel_path

    if not p.exists():
        p = DATA_DIR / "images" / rel_path

    if not p.exists():
        raise FileNotFoundError(f"Image not found: {rel_path}, tried {p}")

    img = Image.open(p).convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
    #img = resize_with_padding(img, IMG_SIZE)
    return img


def build_prompt_with_custom_choices(row, choices):
    metadata = build_metadata(row) if "build_metadata" in globals() else ""

    return build_prompt_from_parts(
        question=row["question"],
        choices=choices,
        answer=None,
        hint=_get_text_field(row, "hint"),
        lecture=_get_text_field(row, "lecture"),
        metadata=metadata,
        solution="",
        include_answer=False,
    )


def make_permutations_for_row(row, k=5):
    choices = ensure_choices(row["choices"])
    n = int(row["num_choices"])

    perms = [list(range(n))]
    rng = random.Random(stable_seed_from_id(row["id"] if "id" in row else str(row.name)))
    seen = {tuple(perms[0])}

    attempts = 0
    while len(perms) < k and attempts < 100:
        perm = list(range(n))
        rng.shuffle(perm)

        if tuple(perm) not in seen:
            perms.append(perm)
            seen.add(tuple(perm))

        attempts += 1

    return perms


# Core prediction functions
@torch.inference_mode()
def predict_batch_shuffle_ensemble(df_batch, k=5):
    all_prompts = []
    all_images = []
    mapping_info = []

    for local_row_idx, (_, row) in enumerate(df_batch.iterrows()):
        choices = ensure_choices(row["choices"])
        n = int(row["num_choices"])
        image = load_image_safe(row["image_path"])

        perms = make_permutations_for_row(row, k=k)

        for perm in perms:
            shuffled_choices = [choices[i] for i in perm]
            prompt = build_prompt_with_custom_choices(row, shuffled_choices)

            all_prompts.append(prompt)
            all_images.append(image)
            mapping_info.append({
                "local_row_idx": local_row_idx,
                "perm": perm,
                "num_choices": n,
            })

    inputs = processor(
    text=all_prompts,
    images=all_images,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=2048,
    )

    inputs = {
        key: value.to(model.device) if torch.is_tensor(value) else value
        for key, value in inputs.items()
    }

    outputs = model(**inputs)
    logits = outputs.logits
    attention_mask = inputs.get("attention_mask", None)

    batch_size = len(df_batch)
    max_choices = max(int(row["num_choices"]) for _, row in df_batch.iterrows())

    score_sums = [
        np.zeros(int(row["num_choices"]), dtype=np.float64)
        for _, row in df_batch.iterrows()
    ]

    score_counts = [
        np.zeros(int(row["num_choices"]), dtype=np.float64)
        for _, row in df_batch.iterrows()
    ]

    for expanded_idx, info in enumerate(mapping_info):
        local_row_idx = info["local_row_idx"]
        perm = info["perm"]
        n = info["num_choices"]

        if attention_mask is not None:
            last_pos = int(attention_mask[expanded_idx].sum().item()) - 1
        else:
            last_pos = logits.shape[1] - 1

        next_log_probs = F.log_softmax(logits[expanded_idx, last_pos, :], dim=-1)

        for shuffled_idx in range(n):
            letter = CHOICE_LETTERS[shuffled_idx]
            token_id = CHOICE_TOKEN_IDS[letter]
            score = float(next_log_probs[token_id].item())

            original_idx = perm[shuffled_idx]

            score_sums[local_row_idx][original_idx] += score
            score_counts[local_row_idx][original_idx] += 1.0

    preds = []
    score_list = []

    for i in range(batch_size):
        avg_scores = score_sums[i] / np.maximum(score_counts[i], 1.0)
        pred = int(np.argmax(avg_scores))

        preds.append(pred)
        score_list.append(avg_scores.tolist())

    return preds, score_list

In [ ]:
# Full validation

def evaluate_full_val_shuffle_ensemble(eval_df, k=5, original_batch_size=4):
    model.eval()

    all_preds = []
    all_golds = []
    all_scores = []

    for start in tqdm(
        range(0, len(eval_df), original_batch_size),
        desc=f"Full val shuffle ensemble k={k}"
    ):
        df_batch = eval_df.iloc[start:start + original_batch_size]

        preds, scores = predict_batch_shuffle_ensemble(df_batch, k=k)

        all_preds.extend(preds)
        all_scores.extend(scores)
        all_golds.extend([int(x) for x in df_batch["answer"].values])

    all_preds = np.array(all_preds)
    all_golds = np.array(all_golds)

    correct = int((all_preds == all_golds).sum())
    total = len(all_golds)
    acc = correct / total

    print("=" * 80)
    print("Full Validation Results with Choice-Shuffle Ensemble")
    print("=" * 80)
    print(f"K_SHUFFLES: {k}")
    print(f"Accuracy: {acc:.4f} ({correct}/{total})")
    print("\nPrediction distribution:")
    print(pd.Series(all_preds).value_counts().sort_index())

    return {
        "accuracy": acc,
        "correct": correct,
        "total": total,
        "preds": all_preds,
        "golds": all_golds,
        "scores": all_scores,
    }


val_shuffle_results = evaluate_full_val_shuffle_ensemble(
    val_df,
    k=K_SHUFFLES,
    original_batch_size=ORIGINAL_BATCH_SIZE,
)

Full val shuffle ensemble k=3: 100%|██████████| 262/262 [40:30<00:00,  9.28s/it]

Full Validation Results with Choice-Shuffle Ensemble
K_SHUFFLES: 3
Accuracy: 0.7920 (830/1048)

Prediction distribution:
0    342
1    384
2    243
3     73
4      6
Name: count, dtype: int64


## 8. Test Generation

In [ ]:
# ── 8g. Generate submission.csv with choice-shuffle ensemble ─────────────────
SUBMISSION_PATH = DATA_DIR / "submissionhint4.csv"

def predict_test_shuffle_ensemble(test_df, k=5, original_batch_size=4):
    model.eval()

    all_preds = []
    all_scores = []

    for start in tqdm(
        range(0, len(test_df), original_batch_size),
        desc=f"Test prediction shuffle ensemble k={k}"
    ):
        df_batch = test_df.iloc[start:start + original_batch_size]

        preds, scores = predict_batch_shuffle_ensemble(df_batch, k=k)

        all_preds.extend(preds)
        all_scores.extend(scores)

    return all_preds, all_scores


test_preds, test_scores = predict_test_shuffle_ensemble(
    test_df,
    k=K_SHUFFLES,
    original_batch_size=ORIGINAL_BATCH_SIZE,
)

submission = pd.DataFrame({
    "id": test_df["id"].values,
    "answer": test_preds,
})

# Format checks
assert len(submission) == len(test_df), "Submission length mismatch!"
assert list(submission.columns) == ["id", "answer"], "Submission columns must be exactly: id, answer"
assert submission["answer"].isna().sum() == 0, "There are NaN answers."

submission["answer"] = submission["answer"].astype(int)

bad = []
for i, row in test_df.iterrows():
    pred = int(submission.iloc[i]["answer"])
    n = int(row["num_choices"])

    if pred < 0 or pred >= n:
        bad.append((row["id"], pred, n))

if bad:
    raise ValueError(f"Invalid predictions found: {bad[:10]}")

print("=" * 80)
print("Submission preview")
print("=" * 80)
print(submission.head(20))

print("\nAnswer distribution:")
print(submission["answer"].value_counts().sort_index())

print("\nTest num_choices distribution:")
print(test_df["num_choices"].value_counts().sort_index())

if submission["answer"].nunique() == 1:
    print("\nWARNING: All predictions are the same. Do NOT submit before debugging.")
    print("First 5 debug scores:")
    for i in range(min(5, len(test_scores))):
        print(test_df.iloc[i]["id"], "scores =", test_scores[i], "pred =", test_preds[i])
else:
    print("\nOK: Predictions are not all the same.")

submission.to_csv(SUBMISSION_PATH, index=False)

print("\nSaved submission to:")
print(SUBMISSION_PATH)

Test prediction shuffle ensemble k=3: 100%|██████████| 252/252 [38:03<00:00,  9.06s/it]

Submission preview
            id  answer
0   test_01750       2
1   test_00128       2
2   test_02891       0
3   test_02425       1
4   test_00930       2
5   test_03725       2
6   test_00009       1
7   test_02880       1
8   test_01208       1
9   test_00619       0
10  test_00948       1
11  test_01462       0
12  test_01554       1
13  test_02339       0
14  test_01907       1
15  test_02728       1
16  test_00347       1
17  test_01998       1
18  test_02971       0
19  test_03429       2

Answer distribution:
answer
0    373
1    362
2    196
3     71
4      6
Name: count, dtype: int64

Test num_choices distribution:
num_choices
2    272
3    438
4    260
5     38
Name: count, dtype: int64

OK: Predictions are not all the same.

Saved submission to:
/content/drive/MyDrive/dl_final/submissionhint.csv
